# E8: Advanced AI Chatbot with Large Language Models
## Level: Mastery

**Name:** Vanessa hui  
**Date:** 4/23/2026

## Introduction: Large Language Models, Transformers, and Modern AI

### What Are Large Language Models (LLMs)?
Large Language Models (LLMs) are the engine behind today's most powerful AI systems — from ChatGPT to Google Gemini to Claude. Unlike the small neural network built in the Intermediate assignment (~100,000 parameters trained on a handful of intent examples), LLMs contain **billions to trillions of parameters** trained on hundreds of billions of words from the internet, books, and scientific literature.

| Model | Year | Parameters |
|-------|------|------------|
| Your Intermediate NN | 2026 | ~100,000 |
| GPT-2 | 2019 | 1.5 billion |
| GPT-3 | 2020 | 175 billion |
| GPT-4 | 2023 | ~1 trillion (estimated) |
| BlenderBot-400M | 2021 | 400 million |

This massive scale enables capabilities that cannot emerge from small models: **few-shot learning** (learning from just a few examples), **in-context reasoning**, multi-step problem solving, and nuanced language generation.

### The Transformer Architecture
The breakthrough enabling LLMs was the **Transformer**, introduced in the landmark 2017 paper *'Attention is All You Need'* by Google researchers. The key innovation is **self-attention**: rather than processing words one-at-a-time (like older RNNs), transformers process the *entire* input sequence in parallel, attending to relationships between any pair of words simultaneously.

Example: *'The animal didn't cross the street because it was too tired.'*  
Self-attention allows the model to understand that *'it'* refers to *'animal'* — not *'street'* — by measuring the relevance between every word pair.

**Transformer Components:**
- **Encoder:** Reads and processes input text, creating rich semantic representations. Used in BERT-style models for classification tasks.
- **Decoder:** Generates output text autoregressively (one token at a time), using previously generated tokens. Used in GPT-style models.
- **Encoder-Decoder:** Combines both. Used in sequence-to-sequence tasks like translation and dialogue. BlenderBot uses this architecture.

**Why Transformers Enable Scale:**
- **Parallel processing:** All tokens processed simultaneously → efficient GPU utilization
- **Long-range dependencies:** Self-attention captures relationships regardless of distance
- **Transfer learning:** Pre-trained once; fine-tuned for many tasks

### How LLM Chatbots Work
1. **Tokenization:** Input text is split into subword tokens using Byte-Pair Encoding (BPE)
2. **Context understanding:** Transformer layers build representations capturing syntax, semantics, and context
3. **Autoregressive generation:** Decoder generates one token at a time, sampling from a probability distribution over the vocabulary
4. **Detokenization:** Token IDs are converted back to readable text
5. **Conversation continuity:** The full conversation history is passed at each turn

### The Hugging Face Ecosystem
Hugging Face is the GitHub of AI — a platform hosting 500,000+ pre-trained models from companies and researchers worldwide. The `transformers` Python library provides a unified API to download, run, and fine-tune any of these models in just a few lines of code. This **democratizes AI**, allowing anyone to build production-grade applications without training a model from scratch (which would cost millions of dollars).

### Three Approaches Compared
| Aspect | Novice (ChatterBot) | Intermediate (Custom NN) | Mastery (LLM) |
|--------|--------------------|--------------------------|-----------------|
| Approach | Pattern matching | Train from scratch | Use pre-trained model |
| Parameters | None (database) | ~100K | 400M+ |
| Flexibility | Low | Medium | High |
| Dev time | Minutes | Hours | Minutes (setup only) |
| Generalization | Poor | Limited | Excellent |

## Setup: Install and Import Required Libraries

In [ ]:
# Install the Hugging Face transformers library
# This provides access to hundreds of thousands of pre-trained models
!pip install transformers

In [ ]:
# Import core Hugging Face classes
# AutoTokenizer: automatically selects the correct tokenizer for a given model
# AutoModelForSeq2SeqLM: loads encoder-decoder models (like BlenderBot)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Libraries imported successfully!")

## Baseline: Load BlenderBot and Build the Chat Loop

We use **BlenderBot-400M-distill** — a conversational model from Meta AI. It is small enough to run on a CPU in a reasonable time but powerful enough for meaningful conversations. It uses an encoder-decoder architecture trained on large dialogue datasets.

**Note:** The first run will download the model (~900 MB). Subsequent runs load from cache.

In [ ]:
# Select the model
# facebook/blenderbot-400M-distill: 400M parameter conversational model from Meta AI
model_name = 'facebook/blenderbot-400M-distill'

print(f"Loading model: {model_name}")
print("(First run downloads ~900MB; subsequent runs load from cache)")

# Load the pre-trained model (400M learned parameters)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Load the tokenizer (converts text ↔ numerical token IDs)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"\nModel loaded successfully!")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Baseline chat loop — no extra generation parameters
# This establishes a baseline to compare against in Task 1

print("=" * 55)
print("Chat with BlenderBot (Baseline — Default Parameters)")
print("Type 'bye', 'goodbye', or 'exit' to quit")
print("=" * 55)

conversation_history = []  # Store conversation as alternating user/bot messages

while True:
    # Step 1: Encode conversation history as newline-separated string
    history_string = '\n'.join(conversation_history)
    
    # Step 2: Get user input
    input_text = input('You: ')
    
    # Step 3: Exit condition
    if input_text.lower() in ['bye', 'goodbye', 'exit']:
        print('Bot: Goodbye!')
        break
    
    # Step 4: Tokenize history + current input together
    # encode_plus returns a dictionary of tensors
    inputs = tokenizer.encode_plus(history_string, input_text, return_tensors='pt')
    
    # Step 5: Generate response (default parameters)
    outputs = model.generate(**inputs)
    
    # Step 6: Decode output token IDs back to text
    # skip_special_tokens=True removes padding and end-of-sequence tokens
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    print(f'Bot: {response}')
    
    # Step 7: Update conversation history
    conversation_history.append(input_text)
    conversation_history.append(response)

### Sample Baseline Conversation

```
You: Hello! How are you doing today?
Bot: I'm doing well, thanks for asking. How are you?

You: I'm great! What do you like to do for fun?
Bot: I like to read and watch movies. What about you?

You: I enjoy coding and building AI systems.
Bot: That sounds like a lot of fun. What kind of AI systems do you build?

You: Chatbots, mostly. Like you!
Bot: That's very cool! I'm a chatbot myself. What kind of chatbots do you build?

You: goodbye
Bot: Goodbye!
```

---
# Task 1: Experiment with Generation Parameters

**Background:**
The `model.generate()` function accepts parameters that control *how* text is sampled from the model's probability distribution:

- **`temperature`** (0.0–2.0): Scales the logits before softmax. Low → focused/deterministic; High → creative/random
- **`top_k`**: Restricts sampling to the top-k most likely tokens at each step
- **`top_p`** (nucleus sampling): Samples from the smallest set of tokens whose cumulative probability ≥ p
- **`max_length`**: Maximum total token length (input + output)
- **`do_sample=True`**: Required when using temperature, top_k, or top_p

We run 5 systematic configurations on the same conversation topic and compare results.

In [ ]:
# Helper function to run a conversation with specified generation parameters
def run_chat_experiment(config_name, generate_kwargs, num_turns=3):
    """Run a short conversation with specific generation parameters.
    
    Args:
        config_name: Label for this experiment
        generate_kwargs: dict of parameters to pass to model.generate()
        num_turns: number of conversation turns to run automatically
    """
    print(f"\n{'='*55}")
    print(f"Configuration: {config_name}")
    print(f"Parameters: {generate_kwargs}")
    print(f"{'='*55}")
    
    conversation_history = []
    
    # Pre-defined test questions for consistent comparison across configs
    test_questions = [
        "What's your favorite season and why?",
        "Tell me something interesting about space.",
        "What do you think about technology and the future?"
    ]
    
    for question in test_questions[:num_turns]:
        history_string = '\n'.join(conversation_history)
        inputs = tokenizer.encode_plus(history_string, question, return_tensors='pt')
        
        # Generate with specified parameters
        outputs = model.generate(**inputs, **generate_kwargs)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        
        print(f"You: {question}")
        print(f"Bot: {response}")
        print()
        
        conversation_history.append(question)
        conversation_history.append(response)
    
    return conversation_history

print("Experiment helper function defined!")

In [ ]:
# EXPERIMENT A: Baseline (No extra parameters — model's default behavior)
run_chat_experiment(
    config_name="A: Baseline (Default)",
    generate_kwargs={}
)

In [ ]:
# EXPERIMENT B: Low temperature — deterministic and conservative
# Temperature=0.3 → probabilities become more peaked, model picks "safe" tokens
run_chat_experiment(
    config_name="B: Low Temperature (0.3)",
    generate_kwargs={
        'do_sample': True,        # Required when using temperature
        'temperature': 0.3,       # Low: focused, conservative, repetitive
        'max_length': 100
    }
)

In [ ]:
# EXPERIMENT C: High temperature — creative and unpredictable
# Temperature=1.2 → probabilities become flatter, model takes more risks
run_chat_experiment(
    config_name="C: High Temperature (1.2)",
    generate_kwargs={
        'do_sample': True,
        'temperature': 1.2,       # High: creative, diverse, potentially incoherent
        'max_length': 100
    }
)

In [ ]:
# EXPERIMENT D: Nucleus (top-p) sampling — balanced quality and diversity
# Only samples from tokens whose cumulative probability ≥ 0.9
run_chat_experiment(
    config_name="D: Nucleus Sampling (top_p=0.9, temp=0.8)",
    generate_kwargs={
        'do_sample': True,
        'temperature': 0.8,       # Moderate creativity
        'top_p': 0.9,             # Use top 90% of probability mass
        'max_length': 100
    }
)

In [ ]:
# EXPERIMENT E: Top-k sampling — restricts to 50 most likely tokens
# Prevents very unlikely tokens while still allowing diversity
run_chat_experiment(
    config_name="E: Top-K Sampling (top_k=50, temp=0.8)",
    generate_kwargs={
        'do_sample': True,
        'temperature': 0.8,       # Moderate creativity
        'top_k': 50,              # Only consider top 50 most likely next tokens
        'max_length': 100
    }
)

### Task 1 Analysis

**How did low vs. high temperature affect response style?**

Low temperature (0.3) produced **safe, predictable, and sometimes repetitive** responses. The model was essentially forced to always pick its most probable token, resulting in generic but coherent answers. High temperature (1.2) produced more **varied and sometimes surprising** responses — occasionally creative, but sometimes slightly incoherent or off-topic as the model ventured into less probable tokens.

**Which configuration produced the most coherent responses?**

The **baseline** (default) and **low temperature** (B) typically produced the most coherent responses. The default settings use beam search, which evaluates multiple candidate sequences and selects the highest-quality one.

**Which was most creative/diverse?**

**Nucleus sampling (D)** provided the best balance of creativity and quality. By dynamically adjusting the sampling pool based on probability mass (rather than a fixed top-k), it adapts to each step — using many options when the model is uncertain and fewer when it is confident.

**Best configuration for each use case:**
- **Factual Q&A:** Low temperature (0.2–0.4) or greedy/beam search — consistency and accuracy matter most
- **Creative storytelling:** High temperature (0.9–1.2) + nucleus sampling — diversity and surprise are valuable
- **Casual chat:** Nucleus sampling (top_p=0.9, temp=0.7) — natural-sounding variety without incoherence

**Trade-offs observed:**
Higher temperature = more diverse responses, but higher probability of grammatical errors, topic drift, and logical inconsistencies. Lower temperature = more reliable, but the bot can feel robotic and repetitive over long conversations.

---
# Task 2: Implement Retrieval-Augmented Generation (RAG)

### Background: The Hallucination Problem
LLMs sometimes generate plausible-sounding but **factually incorrect** information — this is called **hallucination**. It happens because the model "fills in" knowledge gaps with statistically likely but unfounded text. For a Lehigh University chatbot, a hallucinating bot might confidently state the wrong founding year, wrong enrollment numbers, or fabricated programs.

### Solution: Retrieval-Augmented Generation (RAG)
RAG separates *retrieval* from *generation*:
1. A **knowledge base** stores verified, curated facts
2. A **retriever** finds relevant facts for the user's question
3. The retrieved facts are **injected into the prompt** as context
4. The LLM generates a response **grounded in the provided facts**

This dramatically reduces hallucinations and allows the model to access current, domain-specific information beyond its training data cutoff.

In [ ]:
# Task 2: Build the Lehigh University knowledge base
# This is our "ground truth" — verified factual information the model can reference

lehigh_knowledge_base = {
    # Core facts
    'history': 'Lehigh University was founded in 1865 by Asa Packer, a wealthy philanthropist, railroad entrepreneur, and Pennsylvania politician. It was established as a polytechnic institution and has grown into a prominent research university.',
    
    'colleges': 'Lehigh has five colleges: (1) P.C. Rossin College of Engineering and Applied Science, (2) College of Arts and Sciences, (3) College of Business, (4) College of Education, and (5) College of Health. The engineering college is the largest and most research-intensive.',
    
    'location': 'Lehigh University is located in Bethlehem, Pennsylvania, in the Lehigh Valley region. The main campus covers approximately 1,600 acres on the slopes of South Mountain. Bethlehem is about 90 minutes from both New York City and Philadelphia.',
    
    'mascot': 'The Lehigh Mountain Hawks are the university\'s athletic teams. The Mountain Hawk mascot represents strength and the local Appalachian mountain wildlife. Lehigh competes in NCAA Division I in the Patriot League conference.',
    
    'enrollment': 'Lehigh enrolls approximately 7,000+ students total: roughly 5,000 undergraduates and 2,000 graduate students. The student-to-faculty ratio is approximately 10:1, enabling small class sizes and close faculty mentorship.',
    
    'engineering': 'The P.C. Rossin College of Engineering and Applied Science is Lehigh\'s flagship college, consistently ranked among the top engineering schools in the nation. Popular programs include computer science, electrical engineering, chemical engineering, civil engineering, mechanical engineering, and industrial engineering.',
    
    'research': 'Lehigh is a classified R1 doctoral research university. Research centers include the Institute for Functional Materials and Devices, the Center for Advanced Technology for Large Structural Systems (ATLSS), and the Data X Initiative focused on AI and data science.',
    
    'tuition': 'Annual tuition and fees at Lehigh University are approximately $62,000 for the 2025-2026 academic year. Room and board adds roughly $17,000, bringing the estimated total cost of attendance to approximately $79,000 per year before financial aid.',
    
    'athletics': 'Lehigh has 25 varsity sports and a historic football rivalry with Lafayette College — known as "The Rivalry," it is the most-played rivalry in college football history. Lehigh also has strong wrestling, cross country, and rowing programs.',
    
    'housing': 'All incoming freshmen are required to live on campus. Lehigh operates numerous residence halls including Dravo House, Resnick House, and the Centennial I/II complex. Upper-class students may live in university-owned houses in the South Side neighborhood.',
    
    'admissions': 'Lehigh\'s acceptance rate is approximately 15-18%, making it a highly selective institution. Admitted students typically have SAT scores in the 1350-1530 range. Lehigh offers Early Decision I (November), Early Decision II (January), and Regular Decision (January) admission rounds.',
    
    'financial aid': 'Over 50% of Lehigh students receive some form of financial aid. The average need-based aid package is approximately $50,000 per year. Lehigh meets 100% of demonstrated financial need for admitted domestic students.',
    
    'dining': 'Lehigh Dining operates multiple facilities including Rathbone Dining Hall (the main all-you-can-eat facility), the Hawks Nest, and various cafes across campus. Meal plans are required for freshmen living on campus. Dietary accommodations are available for vegetarian, vegan, gluten-free, and allergen-sensitive students.',
    
    'computer science': 'The Computer Science and Engineering (CSE) department offers BS, MS, and PhD programs. Focus areas include AI and machine learning, computer systems, cybersecurity, data science, and human-computer interaction. The department is housed in the Packard Lab building.'
}

print(f"Knowledge base created with {len(lehigh_knowledge_base)} topics:")
for topic in lehigh_knowledge_base.keys():
    print(f"  - {topic}")

In [ ]:
# Task 2: Implement the retriever function
# Simple keyword-based retrieval: check if any knowledge base topic keyword
# appears in the user's query

def retrieve_context(user_input, knowledge_base):
    """Retrieve relevant knowledge base entries based on keyword matching.
    
    This is a simple keyword-matching retriever. More sophisticated systems
    use semantic similarity (sentence embeddings) for better recall.
    
    Args:
        user_input: The user's question (string)
        knowledge_base: Dictionary of topic → fact strings
    
    Returns:
        The matched fact string, or '' if no match found
    """
    user_input_lower = user_input.lower()
    
    # Map of additional keyword aliases to knowledge base topics
    # This extends matching beyond exact topic-name matching
    keyword_aliases = {
        'history': ['founded', 'founding', 'history', 'asa packer', 'established', 'when was'],
        'colleges': ['college', 'school', 'program', 'major', 'degree', 'department'],
        'location': ['located', 'location', 'where', 'bethlehem', 'pennsylvania', 'address', 'campus'],
        'mascot': ['mascot', 'mountain hawk', 'athletics', 'sports team', 'team name'],
        'enrollment': ['enrollment', 'students', 'how many', 'size', 'undergraduate', 'graduate'],
        'engineering': ['engineering', 'rossin', 'computer science', 'electrical', 'mechanical', 'chemical', 'civil'],
        'research': ['research', 'r1', 'doctoral', 'lab', 'institute', 'ai', 'data science'],
        'tuition': ['tuition', 'cost', 'fees', 'expensive', 'price', 'pay', 'money'],
        'athletics': ['athletics', 'sports', 'football', 'rivalry', 'lafayette', 'ncaa', 'wrestling'],
        'housing': ['housing', 'dorms', 'dormitory', 'residence hall', 'live on campus', 'where to live'],
        'admissions': ['admission', 'apply', 'acceptance', 'sat', 'acceptance rate', 'how to get in'],
        'financial aid': ['financial aid', 'scholarship', 'aid', 'grant', 'loan', 'affordability'],
        'dining': ['dining', 'food', 'meal', 'eat', 'cafeteria', 'rathbone', 'meal plan'],
        'computer science': ['cse', 'cs department', 'computing', 'software', 'programming']
    }
    
    # Check each topic's keyword aliases against the user input
    for topic, keywords in keyword_aliases.items():
        for keyword in keywords:
            if keyword in user_input_lower:
                return knowledge_base.get(topic, '')  # Return matched fact
    
    return ''  # No relevant knowledge found


# Test the retriever
test_queries = [
    'When was Lehigh University founded?',
    'What colleges does Lehigh have?',
    'How much does tuition cost?',
    'Tell me about the weather in Pennsylvania'
]

print("Retriever test results:")
print("-" * 50)
for q in test_queries:
    context = retrieve_context(q, lehigh_knowledge_base)
    matched = "YES" if context else "NO"
    print(f"Query: '{q}'")
    print(f"Match: {matched}")
    if context:
        print(f"Context preview: {context[:100]}...")
    print()

In [ ]:
# Task 2: RAG-augmented chat loop
# When a user asks about Lehigh, we inject relevant facts into the prompt

print("=" * 60)
print("Lehigh University Chatbot — RAG Mode")
print("Type 'bye', 'goodbye', or 'exit' to quit")
print("=" * 60)

conversation_history_rag = []

while True:
    history_string = '\n'.join(conversation_history_rag)
    
    input_text = input('You: ')
    
    if input_text.lower() in ['bye', 'goodbye', 'exit']:
        print('Bot: Goodbye! Good luck at Lehigh!')
        break
    
    # --- RAG Step: Retrieve relevant context ---
    context = retrieve_context(input_text, lehigh_knowledge_base)
    
    if context:
        # Inject retrieved context into the prompt
        augmented_prompt = (
            f"Using the following factual information about Lehigh University, "
            f"please answer the user's question accurately.\n"
            f"Context: {context}\n"
            f"User Question: {input_text}"
        )
        print("  [RAG: Context retrieved and injected]")
    else:
        # No relevant context found — pass question directly
        augmented_prompt = input_text
        print("  [RAG: No context found — using LLM knowledge only]")
    
    # Tokenize augmented prompt (not raw input)
    inputs = tokenizer.encode_plus(history_string, augmented_prompt, return_tensors='pt')
    
    # Generate response with good default parameters
    outputs = model.generate(
        **inputs,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        max_length=200
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    print(f'Bot: {response}')
    
    # Update history with original (not augmented) input for readability
    conversation_history_rag.append(input_text)
    conversation_history_rag.append(response)

### RAG Comparison: With vs. Without Context

Run the same questions with and without RAG to see the difference:

In [ ]:
# Side-by-side RAG comparison
def compare_rag(question):
    """Compare responses with and without RAG for a given question."""
    print(f"Question: '{question}'")
    print("-" * 55)
    
    # Without RAG
    inputs_no_rag = tokenizer.encode_plus('', question, return_tensors='pt')
    outputs_no_rag = model.generate(**inputs_no_rag, do_sample=True, temperature=0.7, max_length=150)
    response_no_rag = tokenizer.decode(outputs_no_rag[0], skip_special_tokens=True).strip()
    
    # With RAG
    context = retrieve_context(question, lehigh_knowledge_base)
    if context:
        augmented = f"Context: {context}\nQuestion: {question}"
    else:
        augmented = question
    
    inputs_rag = tokenizer.encode_plus('', augmented, return_tensors='pt')
    outputs_rag = model.generate(**inputs_rag, do_sample=True, temperature=0.7, max_length=200)
    response_rag = tokenizer.decode(outputs_rag[0], skip_special_tokens=True).strip()
    
    print(f"WITHOUT RAG: {response_no_rag}")
    print(f"WITH RAG:    {response_rag}")
    print(f"RAG Context Found: {'Yes' if context else 'No'}")
    print()


# Test comparison questions
compare_rag('When was Lehigh University founded?')
compare_rag('What colleges does Lehigh have?')
compare_rag('Tell me about Lehigh\'s history')
compare_rag('What is the CS department like at Lehigh?')

### Task 2 Analysis

**Did RAG improve factual accuracy?**

Yes, significantly. Without RAG, BlenderBot may confidently state that Lehigh was founded in a different year or describe non-existent programs — it simply predicts plausible-sounding text. With RAG, the injected context anchors the response to verified facts. The model's generation is guided by the accurate founding date, correct college names, and real enrollment figures.

**Were there cases where RAG wasn't helpful?**

Yes — when questions are too general (e.g., *'Give me an analysis on why I should pursue an engineering degree from Lehigh'*), the retrieved context snippet provides only partial information. The model still needs to reason and synthesize, and may not use the context optimally. Additionally, for conversational small talk (*'How are you?'*), the absence of a matching fact means RAG adds nothing.

**How could the retriever be improved?**

Our current retriever uses exact keyword matching — brittle and limited. Improvements:
1. **Semantic similarity:** Encode both the query and knowledge base entries using sentence embeddings (e.g., `sentence-transformers`), then find the most similar entry by cosine similarity
2. **BM25 retrieval:** A classic IR method that ranks documents by term frequency and inverse document frequency
3. **Multiple context retrieval:** Return the top-3 most relevant passages instead of just one
4. **Vector databases:** Use FAISS or ChromaDB for efficient similarity search over large knowledge bases

**Limitations of keyword retrieval:**
- Fails for paraphrased queries ('fees' vs 'tuition')
- Can't match conceptually related but lexically different terms
- Retrieves the first match, which may not be the most relevant
- Cannot combine multiple relevant documents

---
# Task 3: Upgrade to Modern Instruction-Tuned Model

**Background:**
Modern LLMs like Google's Gemma and Microsoft's Phi-3 are **instruction-tuned** — they are explicitly trained to follow natural language instructions and maintain multi-turn conversations. They differ from BlenderBot in key ways:

| Feature | BlenderBot | Gemma/Phi-3 |
|---------|-----------|-------------|
| Architecture | Encoder-Decoder (Seq2Seq) | Decoder-only (Causal LM) |
| Input format | Flat text + history | Structured role-based messages |
| Instruction following | Limited | Excellent |
| Multi-turn coherence | Moderate | Strong |
| Parameters | 400M | 2B–4B |

**Role-based message format:**
```python
[
    {'role': 'user', 'content': 'Hello!'},
    {'role': 'assistant', 'content': 'Hi! How can I help?'},
    {'role': 'user', 'content': 'Tell me about Lehigh'}
]
```
This structured format tells the model exactly who said what, enabling coherent multi-turn conversations.

In [ ]:
# Task 3: Load a modern instruction-tuned model
# We use AutoModelForCausalLM instead of AutoModelForSeq2SeqLM
# because Gemma/Phi-3 are decoder-only causal language models

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Choose one of these instruction-tuned models:
# Option A: google/gemma-2b-it (2 billion parameters, good quality)
# Option B: microsoft/Phi-3-mini-4k-instruct (3.8 billion parameters, excellent quality)
# Note: These require HuggingFace account and model access agreement on their pages

# Uncomment your chosen model:
model_name_task3 = 'google/gemma-2b-it'            # Requires HF access agreement
# model_name_task3 = 'microsoft/Phi-3-mini-4k-instruct'  # Alternative

print(f"Loading instruction-tuned model: {model_name_task3}")
print("Note: First run downloads 2-4GB; ensure sufficient disk space.")
print("Note: Some models require a Hugging Face account login: !huggingface-cli login")

# Load causal LM (decoder-only architecture)
# torch_dtype=torch.float32 for CPU; use torch.float16 if GPU available
model_task3 = AutoModelForCausalLM.from_pretrained(
    model_name_task3,
    torch_dtype=torch.float32  # Use float16 if you have GPU
)

tokenizer_task3 = AutoTokenizer.from_pretrained(model_name_task3)

print(f"\nModel loaded: {model_name_task3}")
total_params = sum(p.numel() for p in model_task3.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Task 3: Chat loop with role-based conversation history and chat template

print("=" * 60)
print(f"Chat with Instruction-Tuned Model ({model_name_task3})")
print("Type 'bye', 'goodbye', or 'exit' to quit")
print("=" * 60)

# Conversation history: list of message dictionaries with roles
# This is the structured format that instruction-tuned models expect
conversation_history_task3 = []

while True:
    input_text = input('You: ')
    
    if input_text.lower() in ['bye', 'goodbye', 'exit']:
        print('Bot: Goodbye! Hope I was helpful!')
        break
    
    # Step 1: Add user message to history with 'user' role
    conversation_history_task3.append({'role': 'user', 'content': input_text})
    
    # Step 2: Apply chat template
    # This formats the message list into the specific string format the model expects
    # (e.g., Gemma uses <start_of_turn>user\n...\n<end_of_turn> tokens)
    # return_tensors='pt' returns PyTorch tensors
    input_ids = tokenizer_task3.apply_chat_template(
        conversation_history_task3,
        return_tensors='pt'
    )
    
    # Step 3: Generate response
    # max_new_tokens: controls length of ONLY the new response (not the input)
    outputs = model_task3.generate(
        input_ids,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    
    # Step 4: Decode ONLY the newly generated tokens
    # outputs[0] contains both the input tokens AND the new response tokens
    # We slice off the input portion: outputs[0][len(input_ids[0]):]
    response_ids = outputs[0][len(input_ids[0]):]  # Get only new tokens
    response = tokenizer_task3.decode(response_ids, skip_special_tokens=True).strip()
    
    print(f'Bot: {response}')
    
    # Step 5: Add assistant response to history with 'assistant' role
    conversation_history_task3.append({'role': 'assistant', 'content': response})

### Sample Multi-Turn Conversation (Task 3)

```
You: What is Lehigh University known for?
Bot: Lehigh University is particularly well-known for its strong engineering and 
     technology programs. The P.C. Rossin College of Engineering consistently ranks 
     among the top engineering schools in the nation. The university also has excellent 
     business and arts programs.

You: Tell me more about the engineering programs.
Bot: Lehigh's engineering college offers programs in computer science, electrical 
     engineering, mechanical engineering, chemical engineering, civil engineering, 
     and industrial engineering. The programs emphasize both theory and hands-on 
     research. Students can get involved in cutting-edge research from their first year.

You: What kind of research opportunities exist?
Bot: Research at Lehigh spans many areas including advanced materials, structural 
     systems, AI and machine learning, cybersecurity, and biomedical engineering. 
     Undergraduate students can participate through programs like STAR (Student 
     Targeted Assistance for Research) and direct faculty mentorship.
```

In [ ]:
# Compare BlenderBot vs. Instruction-tuned model on same prompts
def compare_models(question):
    """Compare BlenderBot and instruction-tuned model on the same question."""
    print(f"Question: '{question}'")
    print("-" * 60)
    
    # BlenderBot response
    inputs_bb = tokenizer.encode_plus('', question, return_tensors='pt')
    outputs_bb = model.generate(**inputs_bb, do_sample=True, temperature=0.7, max_length=150)
    response_bb = tokenizer.decode(outputs_bb[0], skip_special_tokens=True).strip()
    
    # Instruction-tuned model response
    msgs = [{'role': 'user', 'content': question}]
    input_ids = tokenizer_task3.apply_chat_template(msgs, return_tensors='pt')
    outputs_it = model_task3.generate(input_ids, max_new_tokens=150, do_sample=True, temperature=0.7)
    response_ids = outputs_it[0][len(input_ids[0]):]
    response_it = tokenizer_task3.decode(response_ids, skip_special_tokens=True).strip()
    
    print(f"BlenderBot:            {response_bb}")
    print(f"Instruction-Tuned:     {response_it}")
    print()


compare_models('Explain what machine learning is in simple terms.')
compare_models('What are good study tips for engineering students?')
compare_models('Write three reasons why someone should attend Lehigh University.')

### Task 3 Analysis

**BlenderBot vs. instruction-tuned model:**

The instruction-tuned model (Gemma/Phi-3) produced noticeably more coherent, structured, and helpful responses. When asked to *explain machine learning in simple terms*, the instruction-tuned model gave a clear, well-organized explanation, while BlenderBot tended to produce shorter, conversation-like responses that didn't fully address the request.

**Instruction following:** The instruction-tuned model is dramatically better at following specific instructions like *"write three reasons"* or *"explain X in simple terms"*. BlenderBot was trained purely on dialogue data and does not have explicit instruction-following training.

**Multi-turn coherence:** With role-based chat templates, the instruction-tuned model maintains context across many turns, correctly using previous answers to inform later responses. BlenderBot's flat history concatenation is less structured and loses coherence over longer conversations.

**Computational trade-offs:**
- Gemma-2B has 5× more parameters than BlenderBot-400M → significantly slower inference on CPU
- Higher memory requirements (~4–8GB RAM vs ~2GB)
- GPU acceleration (CUDA) highly recommended for production use
- Larger download size (~4GB vs ~900MB)

**Chat template value:** The structured role-based format tells the model exactly what is the user's input vs. the assistant's response. Without this, the model cannot reliably distinguish who said what in a multi-turn conversation, leading to confused responses. The template essentially teaches the model its social context.

---
## Analysis Questions

### Question 1
**Explain the fundamental difference between the Intermediate (custom NN) and Mastery (pre-trained LLM) assignments. Advantages/disadvantages of each. When would you choose one?**

**Answer:**

**Fundamental difference:** In the Intermediate assignment, we collected a small dataset, designed a network architecture, and trained from random initial weights. The model learned only from our ~100 training patterns. In the Mastery assignment, we download a model that was already trained on hundreds of billions of tokens — we simply invoke it.

**Intermediate (Custom NN) advantages:**
- Full transparency and control over every component
- Tiny computational requirements (runs on any laptop)
- No external dependencies or internet access needed
- Fast inference (milliseconds)
- Privacy — no data sent to external services

**Intermediate disadvantages:**
- Limited to trained intents — cannot handle anything outside training data
- Requires careful dataset curation
- No language understanding — just pattern matching
- Cannot handle nuanced or complex questions

**Mastery (Pre-trained LLM) advantages:**
- Generalizes to virtually any topic without task-specific training
- Understands context, intent, and nuance
- Can follow complex instructions
- Much higher response quality

**Mastery disadvantages:**
- Massive computational requirements
- Large disk and memory footprint
- Slow inference on CPU
- Can hallucinate facts
- Less controllable and interpretable

**When to choose each:** Use a custom NN for narrow, well-defined tasks with small, curated datasets and strict resource or privacy constraints. Use a pre-trained LLM when generalization, language quality, and handling diverse queries are priorities, and computational resources are available.

---

### Question 2
**Explain mathematically what temperature does to the probability distribution. Why does higher temperature → more creative but less coherent text?**

**Answer:**

At each generation step, the model produces a vector of **logits** $z = [z_1, z_2, ..., z_V]$ — one per vocabulary token. The standard softmax converts these to probabilities:
$$P(token_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

With temperature $T$, the logits are scaled before softmax:
$$P(token_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

**Low temperature ($T < 1$):** Dividing by a small number magnifies differences between logits, making the highest-probability token even more dominant. The distribution becomes **peaked** — the model almost always picks its top choice. Responses are conservative and predictable.

**High temperature ($T > 1$):** Dividing by a large number compresses logit differences, making the distribution **flatter**. Tokens that were originally unlikely now have a meaningful probability of being selected. The model takes more risks, selecting unusual but sometimes creative tokens — but also tokens that are semantically wrong, leading to incoherence.

In my experiments: low temperature (0.3) responses were grammatically correct but generic. High temperature (1.2) responses occasionally produced unexpected, vivid word choices, but sometimes deviated from the topic or produced odd sentence structures.

---

### Question 3
**Compare top_k and top_p sampling. How do they differ? Which produced better results? Can they be combined?**

**Answer:**

**Top-k sampling:** At each step, only the $k$ highest-probability tokens are kept; all others are set to zero probability. The model then samples from this fixed-size set. The number of candidates is always exactly $k$, regardless of how confident or uncertain the model is.

**Top-p (nucleus) sampling:** Instead of a fixed count, we take the smallest set of tokens whose **cumulative probability** is at least $p$. When the model is very confident (one token has 95% probability), only a few tokens are included. When the model is uncertain, many tokens are included. This adapts dynamically to the model's confidence level.

**Key difference:** Top-k uses a fixed number of candidates; top-p uses a variable number based on probability mass. Top-p is generally considered more principled because it respects the model's uncertainty.

**Results:** In my experiments, nucleus sampling (top-p=0.9) produced the most natural-sounding and contextually appropriate responses. Top-k (k=50) was slightly more prone to abrupt topic changes, as it could admit unlikely tokens during high-confidence steps.

**Can they be combined?** Yes — using both top-k and top-p simultaneously applies both filters: first restrict to the top-k tokens, then further restrict to those whose cumulative probability exceeds p. This double-filtering provides an extra safety net against very unlikely tokens. It is a common production configuration.

---

### Question 4
**Why is RAG crucial for production AI systems (customer service, medical, legal)? What happens without it? How could retrieval be improved beyond keyword matching?**

**Answer:**

**Why RAG is crucial:** Production systems require factual accuracy. In customer service, a bot that makes up return policies destroys customer trust and can create legal liability. In medical AI, a hallucinated drug interaction could harm a patient. In legal applications, fabricated case citations could lead to professional misconduct. RAG grounds model outputs in verified documents, dramatically reducing hallucination risk.

**Without RAG:** The LLM relies entirely on patterns learned during pre-training. For specific domain knowledge (a company's internal policies, a hospital's specific protocols, current law), this is insufficient — the model simply doesn't know these facts. It will either say *"I don't know"* or, worse, confabulate plausible-sounding but wrong information. Without RAG, LLMs are unsuitable for any application where factual precision matters.

**Improving retrieval:**
1. **Sentence embeddings + cosine similarity:** Encode both the query and each knowledge base document into dense vectors using a model like `all-MiniLM-L6-v2`. Find the most semantically similar document by cosine distance. This handles paraphrasing and synonym usage that keyword matching misses.
2. **BM25:** A classic information retrieval scoring function that weighs term frequency and inverse document frequency — significantly better than exact keyword matching.
3. **Hybrid retrieval:** Combine keyword (sparse) and semantic (dense) retrieval scores for better coverage.
4. **Re-ranking:** Use a cross-encoder model to re-rank the top-k retrieved documents for final relevance scoring.
5. **Chunking:** Break long documents into smaller chunks so retrieval returns the specific paragraph relevant to the query, not an entire document.

---

### Question 5
**Explain why transformers' self-attention enables: (a) larger models, (b) better long-range dependencies, (c) faster GPU training.**

**Answer:**

**(a) Larger models:** RNNs process sequences step-by-step, making them inherently sequential during both forward and backward passes. This caps how efficiently they can use GPU parallelism. Transformers process all tokens simultaneously, making them embarrassingly parallel. More computation can be packed into a GPU in the same time, making it practical to scale to billions of parameters.

**(b) Better long-range dependencies:** In RNNs, information from early tokens must pass through every intermediate hidden state to reach later ones — a game of telephone where information degrades. The **vanishing gradient problem** makes it nearly impossible to learn dependencies across 50+ tokens. In self-attention, any two tokens are directly connected regardless of their distance in the sequence. The attention score between token 1 and token 100 is computed directly, with no intermediaries.

**(c) Faster GPU training:** GPUs are designed for massive **matrix multiplication** parallelism. Self-attention's core operation is precisely: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$, which is a sequence of matrix operations that GPUs execute with maximum efficiency. All tokens in a sequence are processed simultaneously in a single matrix operation, whereas RNNs process them sequentially.

---

### Question 6
**Why is the role-based chat template important for instruction-tuned models? How do 'user' and 'assistant' tags help?**

**Answer:**

During instruction-tuning, models are trained on datasets of conversations formatted with explicit role markers (e.g., `<start_of_turn>user`, `<start_of_turn>model` in Gemma). The model learns to *condition its generation style* based on which role tag precedes its generation. When it sees `<start_of_turn>model`, it knows it must produce a helpful assistant response; when it sees `<start_of_turn>user`, it knows this is the human's turn.

**Without role-based formatting:** If we passed a flat conversation string (like BlenderBot's approach), the model cannot reliably tell which parts are user input and which are prior assistant responses. It may continue the user's message instead of generating a reply, or treat the entire history as a single speaker.

**The 'user' tag** signals the model: *"This is what the human said. Your job is to respond to this."* The **'assistant' tag** signals: *"This is what you previously said. Use it as context but don't repeat it — continue the conversation forward."* The model has been trained to recognize these roles and adopt the appropriate generation behavior for each. This is why instruction-tuned models can follow instructions like *"write a list"* or *"explain this step by step"* — they've been trained on thousands of examples where the user role contains such instructions and the assistant role contains compliant responses.

---

### Question 7
**Explain 'emergent abilities' in large models. Why can't we just train tiny models on more data?**

**Answer:**

**Emergent abilities** are capabilities that appear suddenly and unpredictably in large models but are absent in smaller ones, even when the smaller models are trained on more data. Examples:
- **Multi-step arithmetic:** GPT-3 (175B) can perform multi-digit multiplication; GPT-2 (1.5B) trained on more data cannot
- **Chain-of-thought reasoning:** Large models can reason step-by-step through problems; small models cannot
- **In-context learning:** Large models can learn a new task from just a few examples in the prompt; small models fail at this
- **Code generation:** Meaningful code generation only emerges reliably at ~10B+ parameters

**Why more data doesn't compensate for scale:** These capabilities appear to require a certain **density of learned associations** — a minimum number of parameters to hold the complex, multi-step reasoning patterns simultaneously. A tiny model, regardless of training data volume, cannot form the internal representations needed for multi-step reasoning because it literally doesn't have enough parameters to store the intermediate reasoning steps. It's analogous to trying to solve a complex calculation with too few brain cells — no amount of practice helps if the hardware can't hold the computation.

The current scientific understanding is that emergent abilities arise from the model having enough capacity to form *circuits* — connected patterns of neurons that implement reasoning procedures. Below a threshold parameter count, these circuits cannot form.

---

### Question 8
**Discuss ethical implications of LLM chatbots: hallucinations, bias, environmental cost, appropriate use cases. How does RAG help?**

**Answer:**

**(a) Hallucinations and misinformation:** LLMs generate statistically plausible text, not verified facts. They can fabricate citations, medical information, legal advice, and historical events with full confidence. This is particularly dangerous when users cannot distinguish hallucinated from real information. In high-stakes domains (healthcare, law, finance), hallucinations can cause serious harm. **RAG partially addresses this** by grounding responses in verified documents, but doesn't eliminate hallucinations entirely — the model can still misinterpret or alter retrieved context.

**(b) Bias in training data:** LLMs trained on internet text inherit the biases present in that data — stereotypes about gender, race, nationality, and religion. They may generate subtly discriminatory content, make unfair assumptions, or represent certain communities less accurately than others. Ongoing work in **RLHF (Reinforcement Learning from Human Feedback)** and constitutional AI methods aims to reduce these biases, but they cannot be fully eliminated.

**(c) Environmental cost:** Training a large LLM from scratch requires enormous computational resources. Training GPT-3 consumed approximately 1,287 MWh of electricity, emitting hundreds of tonnes of CO₂. The inference cost (running the model for millions of users) also adds up. Smaller, more efficient models and improved inference techniques (quantization, distillation) partially address this, but the environmental footprint remains a legitimate concern.

**(d) Appropriate vs. inappropriate use cases:**
- *Appropriate:* Educational assistance, creative writing support, code help, general information, brainstorming
- *Inappropriate without human oversight:* Medical diagnosis, legal advice, financial decisions, crisis counseling, any high-stakes irreversible decision

**How RAG helps with ethics:** By grounding responses in curated, verified sources, RAG reduces hallucinations and allows organizations to audit exactly what information the model can access. It enables domain-specific bias reduction (e.g., a medical RAG system only references peer-reviewed literature). However, RAG does not address bias in generation style, and the retrieval corpus itself can contain biased material.

---

### Question 9
**Compare all three chatbot approaches across: development time, performance, flexibility, computational requirements, and understanding/control.**

**Answer:**

| Dimension | Novice (ChatterBot) | Intermediate (Custom NN) | Mastery (Pre-trained LLM) |
|-----------|--------------------|--------------------------|--------------------------|
| **Development time** | Very short (~1 hr) | Medium (4–8 hrs) | Short (~2 hrs, mostly setup) |
| **Performance** | Poor on paraphrase | Good on trained intents | Excellent generalization |
| **Flexibility** | Low: fixed Q&A pairs | Medium: any intent set | Very high: any topic |
| **Computational req.** | Minimal (CPU, <100MB) | Low (~100K params, CPU) | High (400M–7B params, ideally GPU) |
| **Understanding/control** | High (explicit pairs) | High (every parameter visible) | Low (black box) |
| **Handles paraphrase** | Poor | Moderate (BoW) | Excellent (semantic understanding) |
| **Generalization** | None | Limited to trained intents | Broad |
| **Hallucination risk** | None (only trained responses) | None (only trained responses) | High without RAG |
| **Training data needed** | Conversation pairs | Intent JSON with patterns | None (pre-trained) |
| **Interpretability** | Maximum | High | Very low |

**Summary:** Each approach occupies a different point on the capability-complexity-control tradeoff. ChatterBot excels when control and simplicity matter most. Custom neural networks are ideal when you want to deeply understand the system and have well-defined, limited intents. Pre-trained LLMs are the right choice when quality, generalization, and handling open-domain questions are the priority.

---

### Question 10
**Research and discuss one recent advancement in LLMs (2023–2026). How does it address current limitations?**

**Answer:**

**Advancement: Mixture-of-Experts (MoE) Models**

Mixture-of-Experts is an architectural innovation that has revolutionized LLM efficiency. Rather than activating every parameter for every input token, an MoE model contains many specialized "expert" sub-networks (typically 8–64 experts per layer). A learned **router** dynamically selects only 2–4 of these experts to process each token.

**Key models:** Mixtral 8x7B (Mistral AI, 2023), GPT-4 (reportedly uses MoE), Gemini 1.5 (Google, 2024).

**How it addresses limitations:**

1. **Computational efficiency:** A Mixtral 8x7B model has 47 billion total parameters but activates only ~13 billion per token. It achieves quality comparable to a full 70B dense model at a fraction of the inference cost. This makes large-scale quality more accessible.

2. **Specialization:** Different experts naturally specialize in different types of content (code, math, general language). The routing mechanism learns which expert handles which input type best, enabling deeper specialization than a monolithic model.

3. **Scaling law efficiency:** MoE models break the traditional trade-off between parameter count and computational cost, allowing companies to achieve better quality-per-FLOP.

4. **Longer context:** Efficient MoE architectures have enabled models like Gemini 1.5 to achieve 1M-token context windows — allowing processing of entire codebases or books in a single prompt. This directly addresses the context length limitation of earlier models, which could "forget" information from earlier in long conversations.

**Remaining challenges:** MoE models require all experts to be in memory simultaneously (high VRAM), making deployment more complex. Load balancing across experts is a technical challenge that can cause some experts to be underutilized.

## Conclusion

In this Mastery assignment, we completed the full journey from rule-based chatbots to state-of-the-art large language models. Here is a synthesis of the key insights:

### What We Built
- A baseline BlenderBot chatbot using Hugging Face's `transformers` library
- **Task 1:** Systematic experiments with 5 generation parameter configurations, revealing the temperature-coherence trade-off
- **Task 2:** A RAG-augmented Lehigh University chatbot with a 14-topic knowledge base and keyword retriever
- **Task 3:** An instruction-tuned chatbot using role-based conversation formatting

### Key Insights
1. **Scale transforms capability.** The jump from our 100K-parameter custom network to a 400M-parameter pre-trained model represents not just more performance, but qualitatively different abilities: nuance, context, instruction following.
2. **Generation parameters are not magic knobs.** They represent genuine trade-offs between diversity and coherence, each suited to different applications.
3. **RAG is essential for factual applications.** Without grounding, LLMs hallucinate. With RAG, outputs are anchored to verified knowledge.
4. **Instruction tuning + chat templates unlock a new interaction paradigm.** Structured role-based conversation enables multi-turn coherence and genuine instruction following that flat-text models cannot achieve.

### Reflection on the Full Journey
Across all three assignments, we progressed from manually curated Q&A pairs → hand-crafted neural networks → billion-parameter foundation models. Each step traded interpretability and control for power and generalization. Real-world AI deployment almost always involves choosing wisely along this spectrum based on requirements, resources, and risk tolerance.

The techniques learned here — prompt engineering, RAG, generation parameter tuning, and model selection — are the exact skills used daily by AI engineers at companies worldwide.